In [ ]:
import numpy as np
import gymnasium as gym
from typing import Dict, Tuple, Optional
from enum import Enum
from gymnasium.error import DependencyNotInstalled
from envs.stowage_gym import StowageEnv
config_dict = {
    "vessel_shape": (4,4,4),
    "yard_shape": (4,8,4),
    "num_containers": 124,
    "group_num": 5,
    "group_placement": "random",
    "seed": 4307,
}
env = StowageEnv(config=config_dict)

# SB3

In [ ]:
from sb3_contrib import MaskablePPO
from sb3_contrib.common.envs import InvalidActionEnvDiscrete
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.maskable.utils import get_action_masks
# This is a drop-in replacement for EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback


# env = InvalidActionEnvDiscrete(dim=80, n_invalid_actions=60)
model = MaskablePPO("MlpPolicy", env, gamma=0.4, seed=32, verbose=1,tensorboard_log="./ppo_stow_tensorboard/")
model.learn(80_000,tb_log_name="second_run")

evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

model.save("ppo_mask")
del model # remove to demonstrate saving and loading

model = MaskablePPO.load("ppo_mask")

obs, _ = env.reset()

In [8]:
from algorithms.A2C import MaskableA2C
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.maskable.utils import get_action_masks
# This is a drop-in replacement for EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback


# env = InvalidActionEnvDiscrete(dim=80, n_invalid_actions=60)
model = MaskableA2C("MlpPolicy", env, gamma=0.4, seed=32, verbose=1)
model.learn(1000)

# evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

# model.save("a2c_mask")
# del model # remove to demonstrate saving and loading

# model = MaskableA2C.load("a2c_mask")

# obs, _ = env.reset()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 32        |
|    ep_rew_mean        | -23.9     |
| time/                 |           |
|    fps                | 197       |
|    iterations         | 100       |
|    time_elapsed       | 2         |
|    total_timesteps    | 500       |
| train/                |           |
|    entropy_loss       | -1.4      |
|    explained_variance | -1.67e-06 |
|    learning_rate      | 0.0007    |
|    n_updates          | 99        |
|    policy_loss        | -0.569    |
|    value_loss         | 0.543     |
-------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 32       |
|    ep_rew_mean        | -17.7    |
| time/                 |          |
|    fps                | 212      |
|    iterations         | 200      |


In [2]:
from algorithms.DQN import MaskableDQN
from sb3_contrib.common.maskable.evaluation import evaluate_policy

# env = InvalidActionEnvDiscrete(dim=80, n_invalid_actions=60)
model = MaskableDQN("MlpPolicy", env, gamma=0.4, seed=32, verbose=1)
model.learn(1000)

evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32       |
|    ep_rew_mean      | -37.2    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 328      |
|    time_elapsed     | 0        |
|    total_timesteps  | 128      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.505    |
|    n_updates        | 6        |
----------------------------------


/home/stowage_env/algorithms/policies.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  action_masks = th.tensor(action_masks, dtype=th.bool).to(logits.device)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32       |
|    ep_rew_mean      | -30.2    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 365      |
|    time_elapsed     | 0        |
|    total_timesteps  | 256      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.272    |
|    n_updates        | 38       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32       |
|    ep_rew_mean      | -27.9    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 12       |
|    fps              | 378      |
|    time_elapsed     | 1        |
|    total_timesteps  | 384      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.559    |
|    n_updates      

(-7.0, 0.0)

# Ray

In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig
env = StowageEnv(config_dict)
config = PPOConfig()
config.environment(env = StowageEnv, env_config=config_dict)
config.env_runners(num_env_runners=1)
config.training(
    gamma=0.9, lr=0.01, kl_coeff=0.3, train_batch_size_per_learner=256
)

# Build a Algorithm object from the config and run 1 training iteration.
algo = config.build()
algo.train()